In [1]:
!pip -q install geopandas shapely pyproj scipy


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
from __future__ import annotations
from pathlib import Path
import math, random, re, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import Point
from shapely.ops import unary_union
from scipy.ndimage import gaussian_filter

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
DEVICE


device(type='cuda')

In [ ]:
INPUT_BASE = Path('/content/drive/MyDrive/Research')
BASE = Path('/content/drive/MyDrive/version_2')
DRIVE_ROOT = Path('/content/drive/MyDrive')
POINTLEVEL_DIR = BASE / 'pointlevel_outputs_v16c_version_2'
OUT_DIR = BASE / 'patrol_optimization_outputs_v16c_version_2'
OUT_DIR.mkdir(parents=True, exist_ok=True)
PATROL_MAP_DIR = OUT_DIR / 'hourly_patrol_maps'
PATROL_MAP_DIR.mkdir(parents=True, exist_ok=True)

PRED_PATH = POINTLEVEL_DIR / 'hourly_cell_predictions.csv'
CELL_PATH = POINTLEVEL_DIR / 'grid_cells.csv'
META_PATH = POINTLEVEL_DIR / 'grid_metadata.json'
SUMMARY_PATH = POINTLEVEL_DIR / 'heatmap_model_summary.csv'

if not PRED_PATH.exists() or not CELL_PATH.exists() or not META_PATH.exists():
    raise FileNotFoundError('Run PointLevel_Heatmaps_v16c_version_2.ipynb first so the saved prediction files exist in /version_2/pointlevel_outputs_v16c_version_2.')

N_PATROLS = 10
GA_POP = 60
GA_GENERATIONS = 40
GA_MUTATION = 0.20
GA_ELITE = 8
GA_SPREAD_PENALTY = 0.90
GA_MOVE_PENALTY = 0.20
MIN_SECTOR_DISTANCE_M = 3000.0

print('POINTLEVEL_DIR =', POINTLEVEL_DIR)
print('OUT_DIR =', OUT_DIR)
print('PRED_PATH =', PRED_PATH)


POINTLEVEL_DIR = /content/drive/MyDrive/version_2/pointlevel_outputs_v16c_version_2
OUT_DIR = /content/drive/MyDrive/version_2/patrol_optimization_outputs_v16c_version_2
PRED_PATH = /content/drive/MyDrive/version_2/pointlevel_outputs_v16c_version_2/hourly_cell_predictions.csv


In [ ]:
def parse_datetime(df):
    if 'occurred_dt' in df.columns:
        dt = pd.to_datetime(df['occurred_dt'], errors='coerce', utc=True)
        if dt.notna().any():
            return dt.dt.tz_convert(None)
    if 'crime_date' in df.columns and 'crime_time' in df.columns:
        dt = pd.to_datetime(df['crime_date'].astype(str) + ' ' + df['crime_time'].astype(str), errors='coerce', utc=True)
        if dt.notna().any():
            return dt.dt.tz_convert(None)
    if 'cmplnt_fr_dt' in df.columns and 'cmplnt_fr_tm' in df.columns:
        dt = pd.to_datetime(df['cmplnt_fr_dt'].astype(str) + ' ' + df['cmplnt_fr_tm'].astype(str), errors='coerce', utc=True)
        return dt.dt.tz_convert(None)
    raise ValueError('No usable datetime columns found.')

def clean_precinct(x):
    if pd.isna(x):
        return None
    s = re.sub(r'[^0-9]', '', str(x))
    return s if s else None

def build_grid(bounds, geom, grid_m):
    minx, miny, maxx, maxy = bounds
    xs = np.arange(minx + grid_m / 2, maxx, grid_m)
    ys = np.arange(miny + grid_m / 2, maxy, grid_m)
    xx, yy = np.meshgrid(xs, ys)
    flat_x = xx.ravel(); flat_y = yy.ravel()
    mask = np.array([geom.contains(Point(x, y)) for x, y in zip(flat_x, flat_y)])
    flat_x = flat_x[mask]; flat_y = flat_y[mask]
    col = ((flat_x - (minx + grid_m / 2)) / grid_m).round().astype(int)
    row = ((flat_y - (miny + grid_m / 2)) / grid_m).round().astype(int)
    return {'x': flat_x, 'y': flat_y, 'row': row, 'col': col, 'cell_id': np.arange(len(flat_x), dtype=np.int64), 'minx': minx, 'miny': miny, 'nrows': len(ys), 'ncols': len(xs), 'grid_m': grid_m}

def assign_cells(x, y, grid):
    col = ((x - (grid['minx'] + grid['grid_m'] / 2)) / grid['grid_m']).round().astype(int)
    row = ((y - (grid['miny'] + grid['grid_m'] / 2)) / grid['grid_m']).round().astype(int)
    key = row.astype(np.int64) * (grid['ncols'] + 1) + col.astype(np.int64)
    gkey = grid['row'].astype(np.int64) * (grid['ncols'] + 1) + grid['col'].astype(np.int64)
    lookup = dict(zip(gkey, grid['cell_id']))
    return np.array([lookup.get(k, -1) for k in key], dtype=np.int64)

def build_knn_adj(coords, k=8):
    d2 = np.sum((coords[:, None, :] - coords[None, :, :]) ** 2, axis=-1)
    idx = np.argsort(d2, axis=1)[:, 1:k+1]
    n = coords.shape[0]
    adj = np.zeros((n, n), dtype=np.float32)
    for i in range(n):
        adj[i, idx[i]] = 1.0
    adj = np.maximum(adj, adj.T); np.fill_diagonal(adj, 1.0)
    deg = adj.sum(axis=1)
    deg_inv_sqrt = np.power(deg, -0.5)
    deg_inv_sqrt[np.isinf(deg_inv_sqrt)] = 0.0
    D = np.diag(deg_inv_sqrt)
    return D @ adj @ D

def build_hour_features(ts):
    hour = ts.hour; dow = ts.dayofweek; month = ts.month
    return pd.DataFrame({'hour_sin': np.sin(2 * math.pi * hour / 24), 'hour_cos': np.cos(2 * math.pi * hour / 24), 'dow_sin': np.sin(2 * math.pi * dow / 7), 'dow_cos': np.cos(2 * math.pi * dow / 7), 'month_sin': np.sin(2 * math.pi * (month - 1) / 12), 'month_cos': np.cos(2 * math.pi * (month - 1) / 12), 'weekend': (dow >= 5).astype(int)}, index=ts)

def sector_id_from_xy(x, y, minx, miny, tile_m):
    tx = ((x - minx) / tile_m).astype(int)
    ty = ((y - miny) / tile_m).astype(int)
    return ty * 100000 + tx

def render_heatmap(arr, mask, title, out_path, vmax=None):
    sm = gaussian_filter(np.nan_to_num(arr, nan=0.0), sigma=KDE_SIGMA)
    sm[~mask] = np.nan
    plt.figure(figsize=(7, 7))
    plt.imshow(sm, origin='lower', cmap='hot', vmin=0.0, vmax=vmax)
    plt.title(title); plt.axis('off'); plt.colorbar(fraction=0.04, pad=0.02)
    plt.savefig(out_path, dpi=150, bbox_inches='tight'); plt.close()

def format_hour_name(ts):
    return ts.strftime('%Y%m%d_%H00')


In [ ]:
pred_df = pd.read_csv(PRED_PATH, parse_dates=['dt'])
cells = pd.read_csv(CELL_PATH)
metadata = json.loads(META_PATH.read_text(encoding='utf-8'))
if SUMMARY_PATH.exists():
    display(pd.read_csv(SUMMARY_PATH))

grid = {
    'minx': float(metadata['minx']),
    'miny': float(metadata['miny']),
    'nrows': int(metadata['nrows']),
    'ncols': int(metadata['ncols']),
    'grid_m': float(metadata['grid_m']),
    'row': cells['row'].to_numpy(dtype=np.int64),
    'col': cells['col'].to_numpy(dtype=np.int64),
}
PATROL_TILE_M = float(metadata['patrol_tile_m'])
KDE_SIGMA = float(metadata['kde_sigma'])
TEST_HOURS = int(metadata['test_hours'])

mask = np.zeros((grid['nrows'], grid['ncols']), dtype=bool)
mask[grid['row'], grid['col']] = True
cell_to_idx = {cid: i for i, cid in enumerate(cells['cell_id'].values)}
hour_list = sorted(pred_df['dt'].unique())
hour_to_idx = {ts: i for i, ts in enumerate(hour_list)}
test_pred = np.zeros((len(hour_list), len(cells)), dtype=np.float32)
for row in pred_df.itertuples(index=False):
    hi = hour_to_idx[row.dt]
    ci = cell_to_idx[row.cell_id]
    test_pred[hi, ci] = float(row.pred_count)

print('Loaded prediction rows:', pred_df.shape)
print('Hour count:', len(hour_list), 'Cell count:', len(cells))


,target_crime,grid_m,patrol_tile_m,lookback_hours,test_hours,kde_sigma,recent_hour_blend,recent_day_blend,test_mae,test_rmse
0,ALL,2000.0,2000.0,72,72,0.85,0.1,0.2,0.044638,0.161863


Loaded prediction rows: (151344, 6)
Hour count: 72 Cell count: 2102


In [ ]:
cells['sector_id'] = sector_id_from_xy(cells['x'].values, cells['y'].values, grid['minx'], grid['miny'], PATROL_TILE_M)
sector_centers = cells.groupby('sector_id')[['x', 'y']].mean().reset_index()
sector_to_cells = cells.groupby('sector_id')['cell_id'].apply(list).to_dict()
sector_ids = sorted(sector_to_cells.keys())
sector_centers = sector_centers.set_index('sector_id').reindex(sector_ids).reset_index()
sector_coords = sector_centers[['x', 'y']].to_numpy(dtype=np.float32)
sector_dist = np.sqrt(((sector_coords[:, None, :] - sector_coords[None, :, :]) ** 2).sum(-1))
sector_risk = np.zeros((TEST_HOURS, len(sector_ids)), dtype=np.float32)
for i, sid in enumerate(sector_ids):
    cell_idx = [cell_to_idx[c] for c in sector_to_cells[sid]]
    sector_risk[:, i] = test_pred[:, cell_idx].sum(axis=1)

def enforce_spacing(chrom, risk_vec):
    target_n = min(N_PATROLS, len(risk_vec))
    ordered = sorted(set(chrom.tolist() if isinstance(chrom, np.ndarray) else chrom), key=lambda idx: float(risk_vec[idx]), reverse=True)
    selected = []
    for idx in ordered:
        if all(sector_dist[idx, kept] >= MIN_SECTOR_DISTANCE_M for kept in selected):
            selected.append(int(idx))
        if len(selected) >= target_n:
            break
    for idx in np.argsort(-risk_vec):
        if int(idx) in selected:
            continue
        if all(sector_dist[int(idx), kept] >= MIN_SECTOR_DISTANCE_M for kept in selected):
            selected.append(int(idx))
        if len(selected) >= target_n:
            break
    if len(selected) < target_n:
        for idx in np.argsort(-risk_vec):
            if int(idx) not in selected:
                selected.append(int(idx))
            if len(selected) >= target_n:
                break
    return np.array(sorted(selected), dtype=np.int64)

def fitness(chrom, risk_vec, prev_chrom=None):
    chrom = enforce_spacing(chrom, risk_vec)
    score = risk_vec[chrom].sum()
    if len(chrom) > 1:
        spread_pen = 0.0
        hard_close = 0
        for i in range(len(chrom)):
            for j in range(i + 1, len(chrom)):
                dij = sector_dist[chrom[i], chrom[j]]
                spread_pen += 1.0 / (dij + 1.0)
                if dij < MIN_SECTOR_DISTANCE_M:
                    hard_close += 1
        score -= GA_SPREAD_PENALTY * spread_pen
        score -= 2.0 * hard_close
    if prev_chrom is not None:
        overlap = len(set(chrom).intersection(set(prev_chrom)))
        score -= GA_MOVE_PENALTY * (N_PATROLS - overlap)
    return float(score)

def init_population(risk_vec):
    probs = risk_vec + 1e-6
    probs = probs / probs.sum()
    pop = []
    for _ in range(GA_POP):
        chrom = np.random.choice(len(risk_vec), size=min(N_PATROLS, len(risk_vec)), replace=False, p=probs)
        pop.append(enforce_spacing(np.array(sorted(chrom), dtype=np.int64), risk_vec))
    return pop

def crossover(a, b, risk_vec):
    pool = list(dict.fromkeys(list(a) + list(b)))
    if len(pool) < N_PATROLS:
        missing = [i for i in np.argsort(-risk_vec) if i not in pool]
        pool.extend(missing[:N_PATROLS - len(pool)])
    chosen = pool[:]
    random.shuffle(chosen)
    chosen = chosen[:N_PATROLS]
    if len(chosen) < N_PATROLS:
        fallback = [i for i in np.argsort(-risk_vec) if i not in chosen]
        chosen.extend(fallback[:N_PATROLS - len(chosen)])
    return enforce_spacing(np.array(sorted(chosen), dtype=np.int64), risk_vec)

def mutate(chrom, risk_vec):
    chrom = chrom.copy()
    if np.random.rand() < GA_MUTATION:
        replace_at = np.random.randint(0, len(chrom))
        candidates = [i for i in range(len(risk_vec)) if i not in chrom]
        if candidates:
            probs = risk_vec[candidates] + 1e-6
            probs = probs / probs.sum()
            chrom[replace_at] = np.random.choice(candidates, p=probs)
    chrom = np.array(sorted(np.unique(chrom)), dtype=np.int64)
    return enforce_spacing(chrom, risk_vec)

def tournament(pop, scores, k=4):
    idx = np.random.choice(len(pop), size=k, replace=False)
    best = idx[np.argmax([scores[i] for i in idx])]
    return pop[best]

hourly_plan = []
prev_best = None
for h, ts in enumerate(hour_list[:TEST_HOURS]):
    risk_vec = sector_risk[h]
    pop = init_population(risk_vec)
    for _ in range(GA_GENERATIONS):
        scores = [fitness(ch, risk_vec, prev_best) for ch in pop]
        elite_idx = np.argsort(scores)[-GA_ELITE:]
        new_pop = [pop[i].copy() for i in elite_idx]
        while len(new_pop) < GA_POP:
            p1 = tournament(pop, scores)
            p2 = tournament(pop, scores)
            child = crossover(p1, p2, risk_vec)
            child = mutate(child, risk_vec)
            new_pop.append(child)
        pop = new_pop[:GA_POP]
    final_scores = [fitness(ch, risk_vec, prev_best) for ch in pop]
    best = pop[int(np.argmax(final_scores))]
    best_score = float(np.max(final_scores))
    prev_best = best.copy()
    for rank, sector_idx in enumerate(best, start=1):
        sid = sector_ids[int(sector_idx)]
        hourly_plan.append({'dt': ts, 'rank': rank, 'sector_id': int(sid), 'sector_risk': float(risk_vec[sector_idx]), 'fitness_score': best_score, 'center_x': float(sector_centers.iloc[sector_idx]['x']), 'center_y': float(sector_centers.iloc[sector_idx]['y'])})
plan_df = pd.DataFrame(hourly_plan)
plan_df.to_csv(OUT_DIR / 'hourly_patrol_plan_ga.csv', index=False)
plan_df.head()


,dt,rank,sector_id,sector_risk,fitness_score,center_x,center_y
0,2024-12-29,1,1900069,0.446102,5.955022,1.052175e+06,159128.369995
1,2024-12-29,2,2300043,0.617484,5.955022,1.000175e+06,167128.369995
2,2024-12-29,3,2600034,0.455808,5.955022,9.821751e+05,173128.369995
3,2024-12-29,4,3300042,0.295642,5.955022,9.981751e+05,187128.369995
4,2024-12-29,5,4200037,0.589960,5.955022,9.881751e+05,205128.369995


In [ ]:
for h, ts in enumerate(hour_list[:TEST_HOURS]):
    img = np.full((grid['nrows'], grid['ncols']), np.nan, dtype=np.float32)
    img[grid['row'], grid['col']] = test_pred[h]
    sm = gaussian_filter(np.nan_to_num(img, nan=0.0), sigma=KDE_SIGMA)
    sm[~mask] = np.nan
    subset = plan_df[plan_df['dt'] == ts]
    plt.figure(figsize=(7, 7))
    plt.imshow(sm, origin='lower', cmap='hot', vmin=0.0, vmax=np.nanpercentile(test_pred, 99))
    sx = ((subset['center_x'].to_numpy() - grid['minx']) / grid['grid_m'])
    sy = ((subset['center_y'].to_numpy() - grid['miny']) / grid['grid_m'])
    plt.scatter(sx, sy, c='cyan', s=40, edgecolors='black')
    for _, r in subset.iterrows():
        px = (r['center_x'] - grid['minx']) / grid['grid_m']
        py = (r['center_y'] - grid['miny']) / grid['grid_m']
        plt.text(px, py, str(int(r['rank'])), color='white', fontsize=7, ha='center', va='center')
    plt.title(f'Predicted Risk + GA Patrol {format_hour_name(ts)}')
    plt.axis('off')
    plt.savefig(PATROL_MAP_DIR / f'patrol_{format_hour_name(ts)}.png', dpi=150, bbox_inches='tight')
    plt.close()
print('Saved patrol plan to', OUT_DIR / 'hourly_patrol_plan_ga.csv')
print('Saved patrol maps to', PATROL_MAP_DIR)


Saved patrol plan to /content/drive/MyDrive/version_2/patrol_optimization_outputs_v16c_version_2/hourly_patrol_plan_ga.csv
Saved patrol maps to /content/drive/MyDrive/version_2/patrol_optimization_outputs_v16c_version_2/hourly_patrol_maps
